# 07. Статистика: оценки, MLE/MAP, A/B-тесты

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/07_statistics.ipynb)

Код из раздела [README.md](https://github.com/justxor/math-for-ai-2026/blob/main/README.md#m07). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### Бутстрап

In [ ]:
import numpy as np
def bootstrap_ci(x, stat=np.median, B=10_000, alpha=0.05):
    rng = np.random.default_rng(0)
    stats = [stat(rng.choice(x, size=len(x), replace=True)) for _ in range(B)]
    return np.quantile(stats, [alpha / 2, 1 - alpha / 2])

latency = np.random.lognormal(3, 0.8, 500)
print(bootstrap_ci(latency))                    # 95% ДИ медианной задержки

### 🏋️ Практика модуля

**7.2.** A: 200 конверсий из 4000, B: 250 из 4000. Значима ли разница на уровне 0.05?

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np
from scipy import stats
xa, na, xb, nb = 200, 4000, 250, 4000
p = (xa + xb) / (na + nb)
z = (xb / nb - xa / na) / np.sqrt(p * (1 - p) * (1 / na + 1 / nb))
print(z, 2 * (1 - stats.norm.cdf(abs(z))))   # z ≈ 2.43, p ≈ 0.015 → значимо

### 🏋️ Практика модуля

**7.3.** Покажите моделированием, что подглядывание раздувает ошибку I рода.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0); false_pos = 0
for _ in range(2000):                          # 2000 A/A-тестов (эффекта нет)
    a = rng.random(14 * 1000) < 0.05; b = rng.random(14 * 1000) < 0.05
    for day in range(1, 15):                   # смотрим каждый день
        n = day * 1000; pa, pb = a[:n].mean(), b[:n].mean(); pp = (pa + pb) / 2
        z = (pb - pa) / np.sqrt(pp * (1 - pp) * 2 / n)
        if abs(z) > 1.96:
            false_pos += 1; break
print(false_pos / 2000)   # ≈ 0.2 вместо 0.05

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def bias_variance():
    rng = np.random.default_rng(3)
    f = lambda x: np.sin(2 * np.pi * x)
    x = np.sort(rng.uniform(0, 1, 15)); y = f(x) + rng.normal(0, 0.25, 15)
    xs = np.linspace(0, 1, 300)
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.6))
    for ax, d, t in zip(axes[:3], [1, 4, 14], ["степень 1: недообучение\n(высокий bias)",
                                                 "степень 4: в самый раз",
                                                 "степень 14: переобучение\n(высокая variance)"]):
        coef = np.polyfit(x, y, d)
        ax.plot(xs, f(xs), color="gray", ls="--", label="истина")
        ax.scatter(x, y, color=C[0], zorder=3)
        ax.plot(xs, np.polyval(coef, xs), color=C[1], lw=2)
        ax.set_ylim(-2, 2); ax.set_title(t, fontsize=10)
    xt = rng.uniform(0, 1, 500); yt = f(xt) + rng.normal(0, 0.25, 500)
    degs = range(0, 15); tr, te = [], []
    for d in degs:
        c = np.polyfit(x, y, d)
        tr.append(np.mean((np.polyval(c, x) - y)**2)); te.append(np.mean((np.polyval(c, xt) - yt)**2))
    axes[3].plot(degs, tr, "o-", color=C[0], label="train"); axes[3].plot(degs, te, "o-", color=C[1], label="test")
    axes[3].set_yscale("log"); axes[3].set_xlabel("сложность модели"); axes[3].legend()
    axes[3].set_title("Ошибка: train ↓ всегда, test — U-образная")
    save(fig, "bias_variance")

bias_variance()